# Supp Fig 3d: clinical RMST with and without CNV cluster 2 as a covariate

**Reconstruction.** No code for this panel was found (repo history on all branches, EFS
Final_Notebooks, Analysis_Repo incl. Branch_*, results/, figures/ incl. zips, other clones);
the supplementary figure image is not in the manuscript PDF either. This notebook implements
the analysis as the Results describe it: "To determine if CNV cluster 2 was driving the
predictive value of previous assessed clinical covariates we reran RMST analysis with and
without cluster 2 as a covariate (Supp Fig. 3d). All previous covariates were unchanged
following correction for cluster 2".

Model: the clinical RMST of Fig 1b / Supp Fig 1c (`fig1b_suppfig1_GBM_MS_Summary.ipynb`,
"RMST Survival Modeling": one row per subject, τ = 3 years, each covariate adjusted for the
others and the centre dummies), run on the patients with a WES CNV cluster, once as it is
and once with a cluster-2 indicator added to the covariates.

Produces:
- **Supp Fig 3d**: forest of the adjusted RMST difference of MGMT, age > 65, sex and p53,
  without and with the cluster-2 covariate.
- The same model on all 287 subjects (reproduces Fig 1b: MGMT +7.1, age > 65 −5.4,
  female +3.1, p53 +3.7 months) and a variant with cluster 2 coded 0 for the patients
  without WES.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gbmcollaboration import AllDataCenters
from gbmcollaboration.paper import config, output  # config sets R_HOME for rpy2
import rpy2.robjects as ro  # noqa: E402

ro.r("suppressPackageStartupMessages({library(survival); library(survRM2)})")


def rmst_adjusted(df, arm, covariates, tau):
    """survRM2 covariate-adjusted RMST difference (arm 1 - arm 0), in months."""
    d = df.dropna(subset=["time", "event", arm] + covariates)
    cov = d[covariates].to_numpy(dtype=float)
    ro.globalenv["t"] = ro.FloatVector(d["time"].to_numpy())
    ro.globalenv["e"] = ro.IntVector(d["event"].astype(int).to_numpy())
    ro.globalenv["a"] = ro.IntVector(d[arm].astype(int).to_numpy())
    ro.globalenv["tau_val"] = float(tau)
    ro.globalenv["cov_mat"] = ro.r.matrix(ro.FloatVector(cov.flatten(order="C")),
                                          nrow=cov.shape[0], ncol=cov.shape[1], byrow=True)
    r = ro.r("rmst2(t, e, a, tau = tau_val, covariates = cov_mat)$adjusted.result[1, c('Est.', 'lower .95', 'upper .95', 'p')]")
    return dict(n=len(d), n_arm1=int(d[arm].sum()), n_arm0=int((d[arm] == 0).sum()),
                diff_months=12 * r[0], ci_low=12 * r[1], ci_high=12 * r[2], p=r[3])

In [ ]:
OUT = output.setup("suppfig3d_wes_cluster_covariate_rmst")
TAU = 3.0
CLUSTER_CSV = config.CNV_CLUSTERS_CSV
# covariate of interest (arm) -> the other covariates of the Fig 1b model
MODELS = {
    "MGMT: methylated vs unmethylated": ("mgmt_arm", ["p53_num", "sex_num", "age_z"]),
    "Age: >65 vs ≤65": ("age_arm", ["mgmt_num", "p53_num", "sex_num"]),
    "Sex: female vs male": ("sex_num", ["mgmt_num", "p53_num", "age_z"]),
    "p53: positive vs negative": ("p53_arm", ["mgmt_num", "sex_num", "age_z"]),
}
PUBLISHED_FIG1B = {"MGMT: methylated vs unmethylated": 7.1, "Age: >65 vs ≤65": -5.4,
                   "Sex: female vs male": 3.1, "p53: positive vs negative": 3.7}

## Clinical table and CNV clusters

As Fig 1b: processed clinical tables of the three centres, one row per subject (baseline
row first). Covariates missing for a patient count as the reference level; the covariate of
interest itself must be known.

In [ ]:
clin = AllDataCenters.load_clinical(exclude_idh1_mut=False)
clin["_priority"] = clin["Sample collection chronology"].map({"Baseline": 0, "At recurrence": 1}).fillna(2)
clin = clin.sort_values(["Subject Id", "_priority"]).drop_duplicates("Subject Id")

df = clin[clin["os_years"].notna() & clin["os_event"].notna()].copy()
df["time"] = pd.to_numeric(df["os_years"], errors="coerce")
df["event"] = pd.to_numeric(df["os_event"], errors="coerce")
df = df[df["time"] > 0].copy()

df["mgmt_arm"] = df["MGMT promoter methylation"].map({"Methylated": 1.0, "Unmethylated": 0.0})
df["p53_arm"] = df["p53"].map({"Positive": 1.0, "Negative": 0.0})
df["age_arm"] = (df["age_at_diagnosis"] > 65).astype(float)
df["mgmt_num"] = (df["MGMT promoter methylation"] == "Methylated").astype(float)
df["p53_num"] = (df["p53"] == "Positive").astype(float)
df["sex_num"] = (df["Administrative gender"] == "Female").astype(float)
df["age_z"] = (df["age_at_diagnosis"] - df["age_at_diagnosis"].mean()) / df["age_at_diagnosis"].std()
center_dummies = pd.get_dummies(df["Site code"].astype(str), prefix="center", drop_first=True).astype(float)
df = df.join(center_dummies)
CENTER_COLS = center_dummies.columns.tolist()

clusters = pd.read_csv(CLUSTER_CSV)
clusters["patient"] = clusters["orig.ident"].str[:8]
clusters = clusters.drop_duplicates("patient").set_index("patient")["cnv_leiden"]
df["patient"] = df["Subject Id"].astype(str).str[:8]
df["cnv_cluster"] = df["patient"].map(clusters)
df["cluster2"] = (df["cnv_cluster"] == 2).astype(float).where(df["cnv_cluster"].notna())

wes = df[df["cluster2"].notna()].copy()
print(f"Subjects with OS: {len(df)}; with a CNV cluster: {len(wes)} "
      f"(cluster 2: {int(wes['cluster2'].sum())}); centres {CENTER_COLS} vs CH")

## RMST with and without the cluster-2 covariate

In [ ]:
rows = []
for label, (arm, covs) in MODELS.items():
    rows.append(dict(covariate=label, cohort="all subjects", model="clinical",
                     **rmst_adjusted(df, arm, covs + CENTER_COLS, TAU)))
    rows.append(dict(covariate=label, cohort="all subjects", model="+ cluster 2 (0 without WES)",
                     **rmst_adjusted(df.assign(cluster2=df["cluster2"].fillna(0)), arm,
                                     covs + CENTER_COLS + ["cluster2"], TAU)))
    for model, extra in [("clinical", []), ("+ cluster 2", ["cluster2"])]:
        rows.append(dict(covariate=label, cohort="CNV-clustered", model=model,
                         **rmst_adjusted(wes, arm, covs + CENTER_COLS + extra, TAU)))
res = pd.DataFrame(rows)
print(res.round(4).to_string(index=False))

In [ ]:
check = res[(res["cohort"] == "all subjects") & (res["model"] == "clinical")].set_index("covariate")["diff_months"]
print("Fig 1b check (all subjects, clinical model):")
for k, v in PUBLISHED_FIG1B.items():
    print(f"  {k:35s} {check[k]:+.1f} months (published {v:+.1f})")

wide = res[res["cohort"] == "CNV-clustered"].pivot(index="covariate", columns="model", values=["diff_months", "p"])
print("\nCNV-clustered patients, without vs with cluster 2:")
print(wide.round(4).to_string())

## Supp Fig 3d

In [ ]:
plot_df = res[res["cohort"] == "CNV-clustered"].copy()
labels = list(MODELS)
colors = {"clinical": "#007ebc", "+ cluster 2": "#fe7a02"}
offsets = {"clinical": -0.15, "+ cluster 2": 0.15}

fig, ax = plt.subplots(figsize=(6.5, 3.6))
for model, sub in plot_df.groupby("model"):
    sub = sub.set_index("covariate").loc[labels]
    y = np.arange(len(labels)) + offsets[model]
    ax.errorbar(sub["diff_months"], y, xerr=[sub["diff_months"] - sub["ci_low"], sub["ci_high"] - sub["diff_months"]],
                fmt="D", ms=5, color=colors[model], capsize=3, lw=1.2,
                label="without cluster 2" if model == "clinical" else "with cluster 2")
    for yi, (d, p) in zip(y, sub[["diff_months", "p"]].to_numpy()):
        ax.text(11.5, yi, f"{d:+.1f} mo, p={p:.3g}", va="center", fontsize=7, color=colors[model], clip_on=False)
ax.axvline(0, color="#7F8C8D", ls="--", lw=0.8)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
ax.invert_yaxis()
ax.set_xlim(-10, 11)
ax.set_xlabel(f"Adjusted RMST difference (months, τ = {TAU:g} years)")
n = int(plot_df["n"].max())
ax.set_title(f"Clinical covariates with and without CNV cluster 2 (CNV-clustered patients, n ≤ {n})", fontsize=9)
ax.legend(fontsize=8, frameon=False, loc="lower left")
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
plt.tight_layout()
output.save(OUT, "suppfig3d_rmst_with_without_cluster2", fig, res, panel="Supp Fig 3d")
plt.show()

In [ ]:
print(output.versions(["numpy", "pandas", "rpy2"], ["survRM2"]).to_string(index=False))